In [1]:
from tokenizers import Tokenizer
import torch
import matplotlib.pyplot as plt

# Funções

In [2]:
def normalizacao(vetor):
    media = vetor.mean(dim=2, keepdim=True)
    var = vetor.var(unbiased=False, dim=2, keepdim=True)
    normalizacao = (vetor - media) / (var + 1e-5) ** 0.5
    return normalizacao

In [17]:
@torch.no_grad()
def gerar_tokens(modelo, sequencia_inicial, quantidade_novos_tokens, tamanho_contexto):
    sequencia = sequencia_inicial.detach().clone()
    for i in range(quantidade_novos_tokens):
        contexto = sequencia[:, -tamanho_contexto:]
        gpt_resultado = modelo(contexto)
        logits_ultima_posicao = gpt_resultado[:, -1, :]
        previsao = torch.argmax(logits_ultima_posicao, dim=-1, keepdim=True)
        sequencia = torch.cat((sequencia, previsao), dim=1)
        print(sequencia)

    return sequencia

In [4]:
def janela_contexto(ids, tamanho_contexto, passo=1):
    janela = [[], []]

    for i in range(0, len(ids) - tamanho_contexto, passo):
        janela[0].append(ids[i : i + tamanho_contexto])
        janela[1].append(ids[i + 1 : i + tamanho_contexto + 1])

    tensor_entradas_treino = torch.tensor(janela[0], dtype=torch.long)
    tensor_alvo_treino = torch.tensor(janela[1], dtype=torch.long)

    return (tensor_entradas_treino, tensor_alvo_treino)

In [5]:
def lotes_agrupamento(listagens, tamanho=1):
    lotes = []
    for i in range(0, len(listagens), tamanho):
        lotes.append(listagens[i : i + tamanho])

    return lotes

In [6]:
def ler_e_tokenizar(caminho_arquivo="./data/train.txt", limit=-1):
    with open(file=caminho_arquivo, encoding="utf-8") as arquivo_treino:
        texto_treino = arquivo_treino.read(limit)
        return tokenizador.encode(texto_treino)

In [7]:
def treinar_epoca(modelo, otimizador, lotes_entrada, lotes_alvo):
    soma_erros = 0.0
    total_tokens = 0
    modelo.train()
    
    for entradas, alvos in zip(lotes_entrada, lotes_alvo):
        otimizador.zero_grad()

        logits = modelo(entradas)
        logits = logits.reshape(-1, logits.shape[-1])

        alvo = alvos.reshape(-1)

        corpus_erro = torch.nn.functional.cross_entropy(logits, alvo)

        corpus_erro.backward()
        otimizador.step()

        soma_erros += corpus_erro.item() * alvos.numel()
        total_tokens += alvos.numel()
        
    return soma_erros / total_tokens

In [8]:
@torch.no_grad()
def avaliar_epoca(modelo, lotes_entrada, lotes_alvo):
    soma_erros = 0.0
    total_tokens = 0
    modelo.eval()

    for entradas, alvos in zip(lotes_entrada, lotes_alvo):
        logits = modelo(entradas)
        logits = logits.reshape(-1, logits.shape[-1])

        alvo = alvos.reshape(-1)

        corpus_erro = torch.nn.functional.cross_entropy(logits, alvo)

        soma_erros += corpus_erro.item() * alvos.numel()
        total_tokens += alvos.numel()

    return soma_erros / total_tokens

In [9]:
def grafico_historico(*historicos):
    for historico in historicos:
        plt.plot(list(range(1, len(historico[1]) + 1)), historico[1], label=historico[0])
                
    plt.xlabel("Época")
    plt.ylabel("Entropia cruzada média")
    plt.legend()
    plt.show()

# Classes

In [10]:
class AtencaoCausal(torch.nn.Module):
    def __init__(self, dimensao_entrada, dimensao_saida=0):
        super().__init__()
        self.dimensao_entrada = dimensao_entrada
        self.dimensao_saida = (
            dimensao_entrada if dimensao_saida <= 0 else dimensao_saida
        )
        self.camada_linear_query = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )
        self.camada_linear_key = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )
        self.camada_linear_value = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )

    def forward(self, x):
        queries = self.camada_linear_query(x)
        keys = self.camada_linear_key(x)
        values = self.camada_linear_value(x)
        return self.atencao_causal(queries, keys, values)

    def calcular_pontuacoes(self, queries, keys):
        dimensao = keys.shape[-1] ** 0.5
        transposed_keys = keys.transpose(-2, -1)
        return (queries @ transposed_keys) / dimensao

    def aplicar_mascara_causal(self, pontuacoes):
        pontuacoes_mascara = pontuacoes.clone()
        for exemplos_index in range(len(pontuacoes_mascara)):
            tokens_tamanho = len(pontuacoes_mascara[exemplos_index])
            for tokens_index in range(tokens_tamanho):
                pontuacoes_mascara[exemplos_index][tokens_index][
                    tokens_index + 1 : tokens_tamanho
                ] = float("-inf")

        return pontuacoes_mascara

    def calcular_pesos(self, pontuacoes_mascaradas):
        return torch.softmax(pontuacoes_mascaradas, dim=-1)

    def combinar_values(self, pesos, values):
        return pesos @ values

    def atencao_causal(self, queries, keys, values):
        tensor_pe = self.calcular_pontuacoes(queries, keys)
        pontuacoes_mascaradas = self.aplicar_mascara_causal(tensor_pe)
        pesos = self.calcular_pesos(pontuacoes_mascaradas)
        contextos = self.combinar_values(pesos, values)
        return pesos, contextos

In [11]:
class AtencaoMultiplasCabecas(torch.nn.Module):
    def __init__(self, dimensao_entrada, dimensao_saida=0, qtd_cabecas=1):
        super().__init__()
        self.dimensao_entrada = dimensao_entrada

        self.dimensao_saida = (
            dimensao_entrada if dimensao_saida <= 0 else dimensao_saida
        )

        if qtd_cabecas <= 0:
            raise ValueError(
                "A quantidade de cabeças precisa ser maior que zero."
            )

        if self.dimensao_saida % qtd_cabecas != 0:
            raise ValueError(
                "A dimensão de saída precisa ser divisível pela quantidade de cabeças."
            )

        self.qtd_cabecas = qtd_cabecas

        self.cabecas = torch.nn.ModuleList(
            [
                AtencaoCausal(
                    dimensao_entrada,
                    self.dimensao_saida // self.qtd_cabecas,
                )
                for _ in range(self.qtd_cabecas)
            ]
        )

        self.saida_linear = torch.nn.Linear(
            self.dimensao_saida,
            self.dimensao_saida,
            bias=False,
        )

    def forward(self, x):
        cabecas_atencao_contextos = []

        for index in range(len(self.cabecas)):
            cabecas_atencao_contextos.append(
                self.cabecas[index](x)[1]
            )

        cabecas_atencao = torch.cat(
            cabecas_atencao_contextos, dim=2
        )

        return self.saida_linear(cabecas_atencao)

In [12]:
class LayerNorm(torch.nn.Module):
    def __init__(self, qtd_componentes):
        super().__init__()
        self.qtd_componentes = qtd_componentes
        self.epsilon = 1e-5
        self.vetor_escala = torch.nn.Parameter(torch.ones(qtd_componentes))
        self.vetor_deslocamento = torch.nn.Parameter(torch.zeros(qtd_componentes))

    def forward(self, x):
        normalizado = self.normalizacao(x)
        resultado = self.vetor_escala * normalizado + self.vetor_deslocamento
        return resultado

    def normalizacao(self, vetor, eixo=-1):
        media = vetor.mean(dim=eixo, keepdim=True)
        var = vetor.var(unbiased=False, dim=eixo, keepdim=True)
        normalizacao = (vetor - media) / (var + self.epsilon) ** 0.5
        return normalizacao

In [13]:
class MLP(torch.nn.Module):
    def __init__(self, dimensao):
        super().__init__()
        self.dimensao = dimensao
        self.expansao = torch.nn.Linear(dimensao, 4 * dimensao)
        self.ativacao = torch.nn.GELU()
        self.projecao_saida = torch.nn.Linear(4 * dimensao, dimensao)

    def forward(self, x):
        expansao_vetor = self.expansao(x)
        ativacao_vetor = self.ativacao(expansao_vetor)
        projecao_saida_vetor = self.projecao_saida(ativacao_vetor)
        return projecao_saida_vetor

In [14]:
class BlocoTransformer(torch.nn.Module):
    def __init__(self, dimensao, qtd_cabecas):
        super().__init__()
        self.norm_atencao = LayerNorm(dimensao)
        self.atencao = AtencaoMultiplasCabecas(dimensao,dimensao, qtd_cabecas)
        self.norm_mlp = LayerNorm(dimensao)
        self.mlp = MLP(dimensao)
        
    def forward(self, x):
        entrada = self.norm_atencao(x)
        entrada_normalizada = self.atencao(entrada)
        residual_atencao = x + entrada_normalizada
        residual_normalizada = self.norm_mlp(residual_atencao)
        saida_mlp = self.mlp(residual_normalizada)
        saida_bloco = residual_atencao + saida_mlp
        return saida_bloco

In [15]:
class GptPequeno(torch.nn.Module):
    def __init__(self, tamanho_vocabulario, tamanho_contexto, dimensao, qtd_cabecas):
        super().__init__()
        self.embedding_tokens = torch.nn.Embedding(tamanho_vocabulario, dimensao)
        self.embedding_posicoes = torch.nn.Embedding(tamanho_contexto, dimensao)
        self.transformer = BlocoTransformer(dimensao, qtd_cabecas)
        self.layer_norm = LayerNorm(dimensao)
        self.projecao_logits = torch.nn.Linear(dimensao, tamanho_vocabulario, bias=False)
    
    def forward(self, x):
        embeddings_tokens = self.embedding_tokens(x)
        indices_posicoes = torch.arange(0, x.shape[1], device=x.device)
        embeddings_posicoes = self.embedding_posicoes(indices_posicoes)
        entrada_transformer = embeddings_tokens + embeddings_posicoes
        saida_transformer = self.transformer(entrada_transformer)
        saida_normalizada = self.layer_norm(saida_transformer)
        logits = self.projecao_logits(saida_normalizada)
        return logits

# 

In [16]:
tokenizador = Tokenizer.from_file("./data/tokenizer.json")

configuracao = {
    "tamanho_vocabulario": 16000,
    "tamanho_contexto": 32,
    "dimensao": 32,
    "qtd_cabecas": 4,
}